# LS-NEQ 10_ttt_injection — Test-Time-Training als strukturelle Injektion

**Dossier-Stufe K** (`lsne.experiments.STAGES["K"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** `docs/DESIGN.md` §4.8 d: fortgesetztes Training eines eingefrorenen Checkpoints auf einem kleinen Kontrastkorpus (Ethereum/DAO-Fork 2016, vom Auftraggeber unter `data/ttt/dao_fork` bereitzustellen) mit drei Kontrollarmen gleichen Budgets (`neutral`: thematisch neutral, gleiche Größe; `nofork`: gleiche Ära ohne Spaltungsnarrativ; `permuted`: Tokens des DAO-Korpus permutiert). Gemessen werden (i) die Memorisationsgeschwindigkeit pro Textstück (Schritte bis zur NLL-Schwelle) über Checkpoints, (ii) Δ Sharing-Index und Δ Netzwerkaktivität und (iii) Δ T(K=8), Δ B(x) sowie Head- und Geometrie-Maße vor/nach dem Training auf demselben Stimulusraster. Das übergebene Modell bleibt unverändert (Kopien); ein Effekt zählt nur, wenn er über allen drei Kontrollarmen liegt.

**Eingaben.** `data/ttt/dao_fork`, `data/ttt/neutral`, `data/ttt/nofork` (*.txt / *.jsonl, siehe `data/ttt/README.md`; fehlen sie, läuft der gemeinfreie Platzhalter mit `is_placeholder = True` — Funktionstest, kein Befund), Stimulusbank, Richtungsbank, SAE (oder Platzhalter), optional Stufe G/I derselben Sitzung (Wolken, schlechtester Head).

**Ausgaben.** Tabellen `ttt_corpora`, `ttt_memorisation`, `ttt_memorisation_boot`, `ttt_training`, `ttt_before_after`, `ttt_effects`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/K/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/10_ttt_injection/`; am Ende eine kurze `summary_10_ttt_injection_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Stufe-K-Parameter (Korpus, Checkpoints) → Modell → [Stufen G, I] → Stufe K → Korpora → Memorisation → Trainingsverlauf → Vorher/Nachher-Effekte → Zusammenfassung.

**Präregistrierung.** Kontrast 17: (i) `ttt_memorisation_boot`, `arm = injection-<Kontrolle>`, `quantity = steps_to_threshold` (zweiseitig); (ii) `ttt_effects`, `measure ∈ {sharing_jaccard, activity_*}`; (iii) `ttt_effects`, `measure ∈ {T_K8, B_lang_dom}` — jeweils `exceeds_all_controls`. Explorativ auf Entwicklungsfamilien; Bestätigung auf Testfamilien mit eingefrorener Schrittzahl und Lernrate.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    # a reused runtime: move the shallow checkout to the branch tip (a fast-forward pull fails on
    # depth-1 history and would silently keep the old code)
    if _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
           f"git -C {REPO_DIR} checkout -q -f -B {GIT_BRANCH} FETCH_HEAD") != 0:
        raise RuntimeError(f"{REPO_DIR} konnte nicht auf den Stand von {GIT_BRANCH} gebracht werden: "
                           "Laufzeit -> Sitzung trennen und löschen, dann erneut ausführen")
elif _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}") != 0:
    raise RuntimeError(f"git clone von {GIT_URL} ({GIT_BRANCH}) fehlgeschlagen")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
# modules imported by an earlier run in this runtime would shadow the updated checkout
for _m in [m for m in sys.modules if m == "lsne" or m.startswith("lsne.")]:
    del sys.modules[_m]
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "10_ttt_injection"
STAGE = "K"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-K-Parameter: Korpus, Kontrollarme, Checkpoints ------------------------------------
TTT_INPUTS = None            # z. B. "/content/drive/MyDrive/LSNE/inputs/ttt": Ordner dao_fork/, neutral/, nofork/ werden nach data/ttt/ kopiert
TTT_CORPUS = None            # Injektionskorpus (Standard data/ttt/dao_fork; Platzhalter mit is_placeholder=True, wenn leer)
TTT_CHECKPOINTS = None       # z. B. ["step4000", "step98000", "step143000"]: Memorisationstabelle über Checkpoints (Shard 7)
WITH_CONTEXT_STAGES = False  # True: Stufen G und I vorher in dieser Sitzung rechnen (Wolken, schlechtester Head -> Vorher/Nachher-Maße)
RUNS_SUBDIR = None           # z. B. "runs_ttt" (Shard 7): eigenes Laufverzeichnis <OUT_ROOT>/runs_ttt statt RUNS_ROOT

import shutil
from dataclasses import replace

if RUNS_SUBDIR:
    _old = RUNS_ROOT
    RUNS_ROOT = os.path.join(OUT_ROOT, RUNS_SUBDIR)
    os.makedirs(RUNS_ROOT, exist_ok=True)
    cfg = replace(cfg, out_root=RUNS_ROOT)
    CLI_COMMAND = CLI_COMMAND.replace(f"--out {_old}", f"--out {RUNS_ROOT}")
    print("Laufverzeichnis:", RUNS_ROOT)
if TTT_INPUTS:
    for arm in ("dao_fork", "neutral", "nofork"):
        src_dir = os.path.join(TTT_INPUTS, arm)
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join("data", "ttt", arm), dirs_exist_ok=True)
            print("kopiert:", src_dir, "-> data/ttt/" + arm)
for arm, p in (("injection", TTT_CORPUS or limits.k_corpus), ("neutral", "data/ttt/neutral"), ("nofork", "data/ttt/nofork")):
    files = [f for f in (os.listdir(p) if os.path.isdir(p) else []) if f.endswith((".txt", ".jsonl"))]
    print(f"{arm:9s} {p}: {len(files)} Datei(en)" + ("" if files else "  -> PLATZHALTER (Funktionstest, kein Befund)"))
if SMOKE:
    TTT_CHECKPOINTS = None
print(f"Training: {limits.k_train_steps} Schritte × Batch {limits.k_train_batch}, lr {limits.k_train_lr}; "
      f"Memorisation: {limits.k_mem_steps} Schritte, lr {limits.k_mem_lr}, Schwelle {limits.k_threshold} nats, "
      f"{limits.k_max_snippets} Stücke/Arm; Checkpoints: {TTT_CHECKPOINTS or 'nur ' + REVISION}; B(x)-Dosis {limits.k_eps}")
if TTT_CORPUS:
    CLI_COMMAND += f" --ttt-corpus {TTT_CORPUS}"
if TTT_CHECKPOINTS:
    CLI_COMMAND += f" --ttt-checkpoints {','.join(TTT_CHECKPOINTS)}"


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def stage_status(stage: str, rd, status: str) -> dict:
    """Report entry of a finished stage: a stale error.txt of an earlier failed run is removed
    (the manifest says ok) and failed substages (G-L, manifest.json -> substages) are listed;
    the status then reads ok_with_substage_errors / reused_with_substage_errors."""
    if C.clear_stale_error(rd.path):
        print(f"Stufe {stage}: veraltete error.txt eines früheren Laufs entfernt ({rd.path})")
    sub = C.substage_status(rd.path, status)
    for name, err in sub["substage_errors"].items():
        print(f"Stufe {stage}: Substufe {name} FEHLER: {err}")
    return sub


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(run_dir=str(rd.path), seconds=0.0, error="", **stage_status(stage, rd, "reused"))
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="",
                             **stage_status(stage, rd, "ok"))
        print(f"Stufe {stage} {report[stage]['status']} in {report[stage]['seconds']:.0f} s: "
              f"{', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe K ausführen (optional nach G und I für die Kontext-Maße) ------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage K --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> \
#       --ttt-corpus data/ttt/dao_fork [--ttt-checkpoints step4000,step98000,step143000]
# (--stage G,I,K additionally yields log_volume / participation_ratio of the clouds and the
# head_delta of the worst head in ttt_before_after, because K reads them from ctx)
if WITH_CONTEXT_STAGES:
    run_stage("G", reuse=False)
    run_stage("I", reuse=False)
res_K = run_stage("K", ttt_corpus=TTT_CORPUS, ttt_checkpoints=TTT_CHECKPOINTS)
print({k: len(v) for k, v in res_K.items()})
manifest_K = substage_report("K")


In [ ]:
# --- Korpora der vier Arme (Platzhalter werden laut markiert) ---------------------------------
with guarded("Korpora"):
    corp = table("K", "ttt_corpora")
    show(corp)
    if len(corp) and truthy(corp["is_placeholder"]).any():
        print("ACHTUNG: mindestens ein Arm läuft auf dem gemeinfreien Platzhalter (is_placeholder = True) -> Funktionstest, "
              "kein Befund. DAO-Fork-Korpus und Kontrollkorpora unter data/ttt/ ablegen (data/ttt/README.md) oder TTT_INPUTS setzen.")


In [ ]:
# --- Kontrast 17 (i): Memorisationsgeschwindigkeit — Schritte bis NLL ≤ Schwelle pro Arm und Checkpoint --
with guarded("Memorisation"):
    mem = table("K", "ttt_memorisation")
    mbt = table("K", "ttt_memorisation_boot")
    arms = [a for a in ["injection", "neutral", "nofork", "permuted"] if a in set(mbt["arm"])]
    titles = {"steps_to_threshold": f"Schritte bis NLL ≤ {limits.k_threshold} (nur erreichte Stücke)",
              "auc_drop": "AUC des NLL-Abfalls", "final_nll": "NLL nach dem Training"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for ax, q in zip(axes, ["steps_to_threshold", "auc_drop", "final_nll"]):
        sub = mbt[(mbt["quantity"] == q) & (mbt["arm"].isin(arms))]
        ck = list(dict.fromkeys(sub["checkpoint"]))
        x = np.arange(len(ck))
        w = 0.8 / max(1, len(arms))
        for i, arm in enumerate(arms):
            g = sub[sub["arm"] == arm].set_index("checkpoint").reindex(ck)
            ax.bar(x + (i - (len(arms) - 1) / 2) * w, g["mean"], w,
                   yerr=[(g["mean"] - g["ci_low"]).clip(lower=0).fillna(0), (g["ci_high"] - g["mean"]).clip(lower=0).fillna(0)],
                   capsize=2, label=arm)
        ax.set_xticks(x)
        ax.set_xticklabels(ck)
        ax.set_title(titles[q], fontsize=9)
    axes[0].legend(fontsize=7)
    savefig("ttt_memorisation")
    plt.show()
    diff = mbt[mbt["arm"].astype(str).str.startswith("injection-")]
    if len(diff):
        print("Injektion minus Kontrolle (gepaart über Stücke, Bootstrap-CI; negativ bei steps_to_threshold = schneller memorisiert):")
        show(diff)
    if len(mem):
        print("Anteil der Stücke, die die Schwelle erreichen, NLL am Start und am Ende pro Arm:")
        show(mem.assign(reached=truthy(mem["reached_threshold"]).astype(float)).groupby(["checkpoint", "arm"])
             .agg(n=("snippet", "count"), reached=("reached", "mean"), nll_start=("nll_start", "mean"), final_nll=("final_nll", "mean")))


In [ ]:
# --- Trainingsverlauf der vier Arme (gleiches Budget: Schritte, Lernrate, Batch, Seed) ----------
with guarded("Trainingsverlauf"):
    tr = table("K", "ttt_training")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    for arm, g in tr.groupby("arm"):
        g = g.sort_values("step")
        a1.plot(g["step"], g["loss"], label=arm)
        a2.plot(g["step"], g["grad_norm"], label=arm)
    a1.set_xlabel("Schritt")
    a1.set_ylabel("Verlust [nats/Token]")
    a1.set_title("Test-Time-Training (AdamW, Gradienten-Clipping 1,0, FP32)", fontsize=9)
    a2.set_xlabel("Schritt")
    a2.set_ylabel("Gradientennorm")
    a2.set_yscale("log")
    a1.legend(fontsize=7)
    savefig("ttt_training")
    plt.show()


In [ ]:
# --- Kontrast 17 (ii)+(iii): Δ nach TTT pro Arm — ein Effekt zählt nur über allen drei Kontrollarmen --
with guarded("Vorher/Nachher"):
    eff = table("K", "ttt_effects")
    corp = table("K", "ttt_corpora")
    arms = [a for a in ["injection", "neutral", "nofork", "permuted"] if a in set(eff["arm"])]
    agg = eff.groupby(["measure", "arm"])["delta"].mean().unstack("arm").reindex(columns=arms)
    order = [m for m in ["T_K8", "B_lang_dom", "logp_target", "sharing_jaccard", "activity_frac_active", "activity_mean_mass",
                         "activity_participation_ratio", "head_delta_mean", "log_volume", "participation_ratio"] if m in agg.index]
    agg = agg.reindex(order)
    fig, axes = plt.subplots(1, len(order), figsize=(2.4 * len(order) + 1, 3.6), squeeze=False)
    for ax, m in zip(axes[0], order):
        ax.bar(range(len(arms)), agg.loc[m].to_numpy(dtype=float), color=["C3" if a == "injection" else "C7" for a in arms])
        ax.set_xticks(range(len(arms)))
        ax.set_xticklabels(arms, rotation=60, fontsize=7)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_title(m, fontsize=8)
    axes[0, 0].set_ylabel("Δ = nach − vor (Mittel über Bedingungen)")
    fig.suptitle("Wirkung des Test-Time-Trainings auf alles Übrige (rot: Injektionsarm, grau: Kontrollarme)", fontsize=10)
    savefig("ttt_effects")
    plt.show()
    inj = eff[eff["arm"] == "injection"]
    if len(inj):
        print("Injektionsarm: liegt der Effekt über allen drei Kontrollen?")
        show(inj[["measure", "condition", "before", "after", "delta", "exceeds_all_controls"]], n=60)
        n_exc = int(truthy(inj["exceeds_all_controls"]).sum())
        note = " (Platzhalterkorpora: identische Arme, erwartungsgemäß 0)" if len(corp) and truthy(corp["is_placeholder"]).all() else ""
        print(f"{n_exc} von {len(inj)} (Maß, Bedingung)-Zeilen über allen Kontrollen{note}")
    missing = [m for m in ("head_delta_mean", "log_volume") if m not in set(eff["measure"])]
    if missing:
        print("Nicht gemessen (Stufen G/I nicht in dieser Sitzung; WITH_CONTEXT_STAGES=True oder CLI --stage G,I,K):", missing)


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    # failed substages (stages G-L) from the manifests: a stage whose table is an empty default
    # must not read plain "ok" in the summary (reused stages included)
    for st, r in report.items():
        if r.get("status") in ("ok", "reused") and r.get("run_dir"):
            r.update(C.substage_status(r["run_dir"], r["status"]))
    substage_errors = {st: r["substage_errors"] for st, r in report.items() if r.get("substage_errors")}
    status_overall = ("error" if any(r.get("status") == "error" for r in report.values())
                      else "substage_errors" if substage_errors else "ok")
    if substage_errors:
        print("ACHTUNG, fehlgeschlagene Substufen:", substage_errors)
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        status_overall=status_overall, substage_errors=substage_errors,
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
